# Alternative Tracing Methods

![AWTT](../../images/alternative_ways_to_trace_0.png)

到目前为止，在本模块中，我们已经了解了可追踪装饰器，以及如何使用它来设置追踪。

在本课中，我们将探讨设置追踪的替代方法，以及何时应该考虑使用这些不同的方法。

## LangChain and LangGraph

如果我们使用 LangChain 或 LangGraph，只需设置几个环境变量即可完成跟踪设置。

![AWTT](../../images/alternative_ways_to_trace_1.png)

In [ ]:
# You can set them inline
import os
# os.environ["OPENAI_API_KEY"] = "your openai api key"
# os.environ["LANGSMITH_API_KEY"] = "your langsmith api key"
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_PROJECT"] = "langsmith-notebook"  # If you don't set this, traces will go to the Default project

In [ ]:
# Or you can use a .env file
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

In [ ]:
import os
import nest_asyncio
import operator
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AnyMessage, get_buffer_string
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display
from typing import List
from typing_extensions import TypedDict, Annotated
from utils import get_vector_db_retriever, RAG_PROMPT

nest_asyncio.apply()

retriever = get_vector_db_retriever()
llm = ChatOpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model_name="qwen3-max", 
    temperature=0
)

# Define Graph state
class GraphState(TypedDict):
    question: str
    messages: Annotated[List[AnyMessage], operator.add]
    documents: List[Document]

# Define Nodes
def retrieve_documents(state: GraphState):
    messages = state.get("messages", [])
    question = state["question"]
    documents = retriever.invoke(f"{get_buffer_string(messages)} {question}")
    return {"documents": documents}

def generate_response(state: GraphState):
    question = state["question"]
    messages = state["messages"]
    documents = state["documents"]
    formatted_docs = "\n\n".join(doc.page_content for doc in documents)
    
    rag_prompt_formatted = RAG_PROMPT.format(context=formatted_docs, conversation=messages, question=question)
    generation = llm.invoke([HumanMessage(content=rag_prompt_formatted)])
    return {"documents": documents, "messages": [HumanMessage(question), generation]}

# Define Graph
graph_builder = StateGraph(GraphState)
graph_builder.add_node("retrieve_documents", retrieve_documents)
graph_builder.add_node("generate_response", generate_response)
graph_builder.add_edge(START, "retrieve_documents")
graph_builder.add_edge("retrieve_documents", "generate_response")
graph_builder.add_edge("generate_response", END)

simple_rag_graph = graph_builder.compile()
display(Image(simple_rag_graph.get_graph().draw_mermaid_png()))

可以通过可选配置 `config={"metadata": {"foo": "bar"}` 传递元数据或其他字段。

In [ ]:
question = "如果我使用 LangChain，该如何设置跟踪？"
simple_rag_graph.invoke({"question": question}, config={"metadata": {"foo": "bar"}})

##### [去 LangSmith 中观察！](https://smith.langchain.com/public/a0391e07-4391-4fee-8fc2-bbb3aeff345f/r) 

## Tracing Context Manager

在 Python 中，您可以使用跟踪上下文管理器将跟踪信息记录到 LangSmith。比 traceable 提供更细粒度控制，这在以下情况下非常有用：

- 你想记录特定代码块的跟踪信息。
- 您希望控制跟踪的输入、输出和其他属性。
- 使用装饰器或包装器是不可行的。

上下文管理器 `trace()` 与 `@traceable` 可追踪装饰器和 `wrap_openai` 包装器无缝集成，因此您可以在同一个应用程序中同时使用它们。

你需要设置 `LANGSMITH_API_KEY` 和 `LANGSMITH_TRACING`

![AWTT](../../images/alternative_ways_to_trace_2.png)

In [ ]:
from langsmith import traceable, trace
from openai import OpenAI
from typing import List
import nest_asyncio
import os
from utils import get_vector_db_retriever

MODEL_PROVIDER = "qwen"
MODEL_NAME = "qwen3-max"
APP_VERSION = 1.0
RAG_SYSTEM_PROMPT = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the latest question in the conversation. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.
"""

openai_client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)
nest_asyncio.apply()
retriever = get_vector_db_retriever()

"""
retrieve_documents
- Returns documents fetched from a vectorstore based on the user's question
"""
@traceable
def retrieve_documents(question: str):
    documents = retriever.invoke(question)
    return documents

"""
generate_response
- Calls `call_openai` to generate a model response after formatting inputs
"""
# TODO: Remove traceable, and use with trace()
# @traceable
def generate_response(question: str, documents):
    # NOTE: Our documents came in as a list of objects, but we just want to log a string
    formatted_docs = "\n\n".join(doc.page_content for doc in documents)

    # TODO: Use with trace()
    with trace(
        name="Generate Response",
        run_type="chain", 
        inputs={"question": question, "formatted_docs": formatted_docs},
        metadata={"foo": "bar"},
    ) as ls_trace:
        messages = [
            {
                "role": "system",
                "content": RAG_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": f"Context: {formatted_docs} \n\n Question: {question}"
            }
        ]
        response = call_openai(messages)
        # TODO: End your trace and write outputs to LangSmith
        ls_trace.end(outputs={"output": response})
    return response

"""
call_openai
- Returns the chat completion output from OpenAI
"""
@traceable
def call_openai(
    messages: List[dict], model: str = MODEL_NAME, temperature: float = 0.0
) -> str:
    response = openai_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response

"""
langsmith_rag
- Calls `retrieve_documents` to fetch documents
- Calls `generate_response` to generate a response based on the fetched documents
- Returns the model response
"""
@traceable
def langsmith_rag(question: str):
    documents = retrieve_documents(question)
    response = generate_response(question, documents)
    return response.choices[0].message.content


In [ ]:
question = "How do I trace with tracing context?"
ai_answer = langsmith_rag(question)
print(ai_answer)

## wrap_openai

Python/TypeScript 中的 `wrap_openai` / `wrapOpenAI` 方法允许您封装 OpenAI 客户端，从而自动记录跟踪信息——无需任何装饰器或函数封装！该封装器可与 @traceable 装饰器或 traceable 函数无缝协作，并且您可以在同一个应用程序中同时使用。

wrap_openai 是专门为那些已经有现成代码直接调用 OpenAI SDK 的用户准备的

你需要设置你的 `LANGSMITH_API_KEY` and `LANGSMITH_TRACING`

![AWTT](../../images/alternative_ways_to_trace_3.png)

In [ ]:
# TODO: Import wrap_openai
from langsmith.wrappers import wrap_openai
import openai
from typing import List
import nest_asyncio
from utils import get_vector_db_retriever

MODEL_PROVIDER = "qwen"
MODEL_NAME = "qwen3-max"
APP_VERSION = 1.0
RAG_SYSTEM_PROMPT = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the latest question in the conversation. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.
"""

# TODO: Wrap the OpenAI Client
openai_client = wrap_openai(
    openai.Client(
        api_key=os.getenv("DASHSCOPE_API_KEY"),
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    )
)

nest_asyncio.apply()
retriever = get_vector_db_retriever()

@traceable(run_type="chain")
def retrieve_documents(question: str):
    return retriever.invoke(question)

@traceable(run_type="chain")
def generate_response(question: str, documents):
    formatted_docs = "\n\n".join(doc.page_content for doc in documents)
    messages = [
        {
            "role": "system",
            "content": RAG_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"Context: {formatted_docs} \n\n Question: {question}"
        }
    ]
    # TODO: We don't need to use @traceable on a nested function call anymore,
    # wrap_openai takes care of this for us
    return call_openai(messages)

# @traceable
def call_openai(
    messages: List[dict],
) -> str:
    return openai_client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
    )

@traceable(run_type="chain")
def langsmith_rag_with_wrap_openai(question: str):
    documents = retrieve_documents(question)
    response = generate_response(question, documents)
    return response.choices[0].message.content


In [ ]:
question = "How do I trace with wrap_openai?"
ai_answer = langsmith_rag_with_wrap_openai(question)
print(ai_answer)

使用 wrap_openai 包装后的 OpenAI 客户端接受与 @traceable 装饰函数相同的 langsmith_extra 参数。

In [ ]:
messages = [
    {
        "role": "user",
        "content": "What color is the sky?"
    }
]

openai_client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
    langsmith_extra={"metadata": {"foo": "bar"}},
)

## [Advanced] RunTree

另一种更明确地将跟踪信息记录到 LangSmith 的方法是使用 RunTree API。此 API 允许您更好地控制跟踪过程——您可以手动创建运行和子运行来组装跟踪信息。您仍然需要设置 `LANGSMITH_API_KEY`，但此方法不需要 `LANGSMITH_TRACING`。

![AWTT](../../images/alternative_ways_to_trace_4.png)

In [ ]:
# import os
# os.environ["OPENAI_API_KEY"] = "your openai api key"
# os.environ["LANGSMITH_API_KEY"] = "your langsmith api key"
# os.environ["LANGSMITH_PROJECT"] = "langsmith-notebook"

In [1]:
from dotenv import load_dotenv
# I have my env variables defined in a .env file
load_dotenv(dotenv_path="../../.env", override=True)

True

我们将 `LANGSMITH_TRACING` 设置为 false，因为在这种情况下，我们将使用 RunTree 手动创建运行。

In [2]:
import os
os.environ["LANGSMITH_TRACING"] = "false"

from langsmith import utils
utils.tracing_is_enabled() # This should return false

False

重写 RAG 应用程序，这次我们在函数调用中传递了一个 RunTree 参数，并在每一层创建子运行。这使得我们的 RunTree 拥有了与之前使用 @traceable 自动建立的相同的层级结构。

In [3]:
from langsmith import RunTree
from openai import OpenAI
from typing import List
import nest_asyncio
from utils import get_vector_db_retriever

openai_client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)
nest_asyncio.apply()
retriever = get_vector_db_retriever()

def retrieve_documents(parent_run: RunTree, question: str):
    # Create a child run
    child_run = parent_run.create_child(
        name="Retrieve Documents",
        run_type="retriever",
        inputs={"question": question},
    )
    documents = retriever.invoke(question)
    # Post the output of our child run
    child_run.end(outputs={"documents": documents})
    child_run.post()
    return documents

def generate_response(parent_run: RunTree, question: str, documents):
    formatted_docs = "\n\n".join(doc.page_content for doc in documents)
    rag_system_prompt = """You are an assistant for question-answering tasks. 
    Use the following pieces of retrieved context to answer the latest question in the conversation. 
    If you don't know the answer, just say that you don't know. 
    Use three sentences maximum and keep the answer concise.
    """
    # Create a child run
    child_run = parent_run.create_child(
        name="Generate Response",
        run_type="chain",
        inputs={"question": question, "documents": documents},
    )
    messages = [
        {
            "role": "system",
            "content": rag_system_prompt
        },
        {
            "role": "user",
            "content": f"Context: {formatted_docs} \n\n Question: {question}"
        }
    ]
    openai_response = call_openai(child_run, messages)
    # Post the output of our child run
    child_run.end(outputs={"openai_response": openai_response})
    child_run.post()
    return openai_response

def call_openai(
    parent_run: RunTree, messages: List[dict], model: str = "qwen3-max", temperature: float = 0.0
) -> str:
    # Create a child run
    child_run = parent_run.create_child(
        name="OpenAI Call",
        run_type="llm",
        inputs={"messages": messages},
    )
    openai_response = openai_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    # Post the output of our child run
    child_run.end(outputs={"openai_response": openai_response})
    child_run.post()
    return openai_response

def langsmith_rag(question: str):
    # Create a root RunTree
    root_run_tree = RunTree(
        name="Chat Pipeline",
        run_type="chain",
        inputs={"question": question}
    )

    # Pass our RunTree into the nested function calls
    documents = retrieve_documents(root_run_tree, question)
    response = generate_response(root_run_tree, question, documents)
    output = response.choices[0].message.content

    # Post our final output
    root_run_tree.end(outputs={"generation": output})
    root_run_tree.post()
    return output

In [4]:
question = "How can I trace with RunTree?"
ai_answer = langsmith_rag(question)
print(ai_answer)

You can trace with RunTree by manually creating a top-level run using the `RunTree` class, then posting it to LangSmith. Use the `withRunTree` helper to propagate the run context to nested or traced functions. This method gives you fine-grained control over tracing but requires careful handling to avoid errors in context propagation.
